# Desktopia on Google Colab — 3D Slicer over a CPU (software) stream

Runs Desktopia's **software path** (no GPU): 3D Slicer renders with Mesa **llvmpipe** under **Xvfb**,
GStreamer captures it with `ximagesrc`, software-encodes **H.264 (x264)**, and streams it over a
**WebSocket** into the cell below. Mouse/keyboard go back via XTEST. Good for slice viewing,
segmentation overlays, and (slow) 3D.

**Runtime:** a plain CPU runtime is fine — no GPU needed. **Browser:** use Chrome (WebCodecs +
WebSocket).

> Colab has no public UDP, so the QUIC/WebTransport path can't be used here; this notebook forces
> the WebSocket transport and reaches it through Colab's port proxy.


## 1. Install dependencies (~1 min)

Minimal set: Xvfb + llvmpipe, the four GStreamer plugin packages the pipeline actually needs
(`ximagesrc`/`videoconvert`/`x264enc`/`h264parse`/`appsink`), and Slicer's Qt + QtWebEngine runtime libs.


In [ ]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq --no-install-recommends \
  xvfb xterm openbox wmctrl xclip libgl1-mesa-dri libglu1-mesa \
  gstreamer1.0-plugins-base gstreamer1.0-plugins-good gstreamer1.0-plugins-bad gstreamer1.0-plugins-ugly \
  python3-gi gir1.2-gstreamer-1.0 gir1.2-gst-plugins-base-1.0 python3-xlib \
  libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-randr0 libxcb-render-util0 libxcb-shape0 \
  libxcb-sync1 libxcb-xfixes0 libxcb-xinerama0 libxcb-xkb1 libxkbcommon-x11-0 libxcb-cursor0 libxcb-util1 \
  libodbc2 libpq5 libpulse-mainloop-glib0 libpcre2-16-0 \
  libxcomposite1 libxdamage1 libxtst6 libhwloc15 libnspr4 libnss3 >/dev/null
# libasound2/libcups2 were renamed *t64 in Ubuntu 24.04; Colab is 22.04 -- install whichever exists
apt-get install -y -qq --no-install-recommends libasound2 libcups2 >/dev/null 2>&1 \
  || apt-get install -y -qq --no-install-recommends libasound2t64 libcups2t64 >/dev/null
pip install -q websockets aioquic   # aioquic only so server.py imports; QUIC is unused on Colab
echo 'deps installed'

## 2. Get Desktopia + 3D Slicer


In [ ]:
%%bash
set -e
REPO=${DESKTOPIA_REPO:-https://github.com/pieper/desktopia}
BRANCH=${DESKTOPIA_BRANCH:-software-render}
rm -rf /content/desktopia
git clone -q --branch "$BRANCH" "$REPO" /content/desktopia || git clone -q "$REPO" /content/desktopia
if ! ls -d /opt/Slicer-*/ >/dev/null 2>&1; then
  echo 'downloading 3D Slicer (~400 MB)...'
  curl -L --retry 3 'https://download.slicer.org/download?os=linux&stability=release' | tar -xz -C /opt
fi
ls -d /opt/Slicer-*/

## 3. Launch Xvfb + Slicer + the streaming server (software mode)

Small geometry (1280×720@15, 4 Mbit) so CPU encoding keeps up. Re-run this cell to restart the stack.


In [ ]:
import os, time, glob, pathlib, subprocess
os.chdir('/content/desktopia')

W, H, FPS, BR = 1280, 720, 15, 4000
env = dict(os.environ, DISPLAY=':2', LIBGL_ALWAYS_SOFTWARE='1', GALLIUM_DRIVER='llvmpipe', HOME='/root')

for pat in ('server.py', 'SlicerApp-real'):
    subprocess.run(['pkill', '-f', pat], check=False)
subprocess.run(['pkill', '-x', 'Xvfb'], check=False); subprocess.run(['pkill', '-x', 'openbox'], check=False)
time.sleep(1)

# self-signed cert (server.py's QUIC config needs one even though we only use WebSocket here)
subprocess.run('openssl req -x509 -newkey ec -pkeyopt ec_paramgen_curve:prime256v1 '
               '-keyout /tmp/k.pem -out /tmp/c.pem -days 1 -nodes -subj /CN=desktopia',
               shell=True, check=True, stderr=subprocess.DEVNULL)

# Xvfb (software framebuffer)
subprocess.Popen(f'Xvfb :2 -screen 0 {W}x{H}x24 +extension GLX +render -noreset',
                 shell=True, env=env, stdout=open('/tmp/xvfb.log','w'), stderr=subprocess.STDOUT)
for _ in range(80):
    if os.path.exists('/tmp/.X11-unix/X2'): break
    time.sleep(0.25)

# openbox: Desktopia rc.xml (single desktop -> wheel reaches Slicer, not workspace-switch) + a menu
os.makedirs('/root/.config/openbox', exist_ok=True)
subprocess.run('cp resources/openbox-rc.xml /root/.config/openbox/rc.xml', shell=True, check=False)
pathlib.Path('/root/.config/openbox/menu.xml').write_text('<?xml version="1.0" encoding="UTF-8"?>\n<openbox_menu xmlns="http://openbox.org/3.4/menu">\n  <menu id="root-menu" label="Desktopia">\n    <item label="Terminal"><action name="Execute"><command>xterm</command></action></item>\n    <item label="3D Slicer"><action name="Execute"><command>sh -c &apos;D=$(ls -d /opt/Slicer-*/ | head -1); exec "$D/Slicer" --no-splash&apos;</command></action></item>\n  </menu>\n</openbox_menu>\n')
subprocess.Popen('openbox', shell=True, env=env, stdout=open('/tmp/wm.log','w'), stderr=subprocess.STDOUT)
time.sleep(1)

# Slicer, then maximize its window once it appears
SDIR = sorted(glob.glob('/opt/Slicer-*/'))[0]
subprocess.Popen(f'{SDIR}/Slicer --no-splash', shell=True, env=env,
                 stdout=open('/tmp/slicer.log','w'), stderr=subprocess.STDOUT)
subprocess.Popen('bash -c "for i in $(seq 1 40); do sleep 2; '
                 'wmctrl -l 2>/dev/null | grep -qi slicer && '
                 '{ wmctrl -r Slicer -b add,maximized_vert,maximized_horz; break; }; done"',
                 shell=True, env=env)

# tell the client page to use the WebSocket transport, then start the server (page+WS on one port)
pathlib.Path('client/status.json').write_text('{"ready":true,"transport":"websocket"}')
subprocess.Popen('python3 server.py --cert /tmp/c.pem --key /tmp/k.pem '
                 f'--source xvfb --width {W} --height {H} --fps {FPS} --bitrate {BR} '
                 '--ws-plain --serve-dir client',
                 shell=True, env=env, stdout=open('/tmp/server.log','w'), stderr=subprocess.STDOUT)
time.sleep(8)
print('--- server.log ---'); print(open('/tmp/server.log').read()[-1500:])
print('Slicer is still loading in the background; the view appears in the cell below shortly.')

## 4. Open the desktop in this cell

Click into the frame to give it focus, then use the mouse/keyboard normally. Right-click the desktop
for a Terminal. Slicer may take a few more seconds to finish loading after the frame connects.


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(4434, path='/index.html', height=760, cache_in_notebook=False)

### Troubleshooting

- **Check Slicer's log:** `print(open('/tmp/slicer.log').read())`
- **Frame stuck on "waiting for launcher…" / "Connecting":** WebSocket may not tunnel through the
  iframe proxy — open it in a tab instead:
  ```python
  from google.colab import output
  output.serve_kernel_port_as_window(4434, path='/index.html')
  ```
- **Black 3D / no slices:** `!glxinfo -B` after `!apt-get install -y mesa-utils` should say *llvmpipe*.
- **Stream never starts:** check `/tmp/server.log` for a GStreamer error.
